# Experiment 9: Create Frontend to Visualize Predictions Using Test Data

**Objective:**
- Create a frontend dashboard to visualize predictions
- Display charts and tables with prediction results
- Integrate with the FastAPI backend

**Prerequisites:** Run Experiments 1-4 (API should be running)

## Step 1: Install Required Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly requests streamlit

## Step 2: Import Libraries and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported!")

## Step 3: Load Test Data and Get Predictions

In [ ]:
# Load dataset
df = pd.read_csv('Bank_Churn_Classification_Dataset.csv', index_col=0)

# Load model artifacts for local predictions
with open('model_artifacts/churn_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('model_artifacts/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('model_artifacts/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

# Preprocess
df_processed = df.drop('CustomerID', axis=1).copy()
for col in label_encoders:
    if col in df_processed.columns:
        df_processed[col] = label_encoders[col].transform(df_processed[col])

X = df_processed.drop('Churn', axis=1)
y = df_processed['Churn']
X_scaled = scaler.transform(X)

# Get predictions
predictions = model.predict(X_scaled)
probabilities = model.predict_proba(X_scaled)

# Add predictions to dataframe
df['Predicted_Churn'] = predictions
df['Churn_Probability'] = probabilities[:, 1]
df['No_Churn_Probability'] = probabilities[:, 0]
df['Prediction_Correct'] = (df['Churn'] == df['Predicted_Churn']).astype(int)

print(f"Dataset: {df.shape[0]} customers")
print(f"Accuracy: {df['Prediction_Correct'].mean():.4f}")
df.head()

## Step 4: API Integration - Batch Predictions

In [ ]:
# Try to get predictions via API (if running)
API_URL = "http://localhost:8000"
API_KEY = "mlops-api-key-001"

def get_api_prediction(row):
    """Get prediction from API for a single customer."""
    try:
        payload = {
            "Gender": row['Gender'],
            "SeniorCitizen": int(row['SeniorCitizen']),
            "Tenure": int(row['Tenure']),
            "MonthlyCharges": float(row['MonthlyCharges']),
            "Contract": row['Contract'],
            "PaymentMethod": row['PaymentMethod'],
            "TotalCharges": float(row['TotalCharges'])
        }
        response = requests.post(
            f"{API_URL}/predict",
            json=payload,
            headers={"X-API-Key": API_KEY},
            timeout=5
        )
        return response.json()
    except:
        return None

# Test API integration with first 5 rows
print("Testing API integration (first 5 customers):")
api_results = []
for idx, row in df.head(5).iterrows():
    result = get_api_prediction(row)
    if result:
        api_results.append(result)
        print(f"  Customer {row['CustomerID']}: {result.get('prediction_label', 'N/A')} (prob: {result.get('churn_probability', 'N/A')})")
    else:
        print(f"  Customer {row['CustomerID']}: API not reachable (using local predictions)")
        break

if not api_results:
    print("\n  Note: Using local model predictions for visualization")

## Step 5: Visualization - Prediction Distribution

In [ ]:
# Prediction Distribution
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Actual Churn Distribution', 'Predicted Churn Distribution'),
                    specs=[[{'type': 'pie'}, {'type': 'pie'}]])

actual_counts = df['Churn'].value_counts()
predicted_counts = df['Predicted_Churn'].value_counts()

fig.add_trace(go.Pie(labels=['No Churn', 'Churn'], values=actual_counts.values,
                     marker_colors=['#2ecc71', '#e74c3c'], name='Actual'), row=1, col=1)
fig.add_trace(go.Pie(labels=['No Churn', 'Churn'], values=predicted_counts.values,
                     marker_colors=['#3498db', '#e67e22'], name='Predicted'), row=1, col=2)

fig.update_layout(title_text='Churn Distribution: Actual vs Predicted', height=400)
fig.show()

In [ ]:
# Churn Probability Distribution
fig = px.histogram(df, x='Churn_Probability', nbins=50, 
                   color=df['Churn'].map({0: 'No Churn', 1: 'Churn'}),
                   color_discrete_map={'No Churn': '#2ecc71', 'Churn': '#e74c3c'},
                   title='Distribution of Churn Probability',
                   labels={'color': 'Actual', 'Churn_Probability': 'Churn Probability'})
fig.update_layout(height=400)
fig.show()

## Step 6: Visualization - Feature Analysis

In [ ]:
# Churn by Contract Type
fig = px.histogram(df, x='Contract', color=df['Predicted_Churn'].map({0: 'No Churn', 1: 'Churn'}),
                   barmode='group', title='Predicted Churn by Contract Type',
                   color_discrete_map={'No Churn': '#2ecc71', 'Churn': '#e74c3c'},
                   labels={'color': 'Prediction'})
fig.update_layout(height=400)
fig.show()

In [ ]:
# Churn Probability by Tenure
fig = px.scatter(df.sample(1000, random_state=42), x='Tenure', y='Churn_Probability',
                 color='Contract', size='MonthlyCharges',
                 title='Churn Probability vs Tenure (sample of 1000)',
                 labels={'Churn_Probability': 'Churn Probability', 'Tenure': 'Tenure (months)'},
                 hover_data=['Gender', 'PaymentMethod', 'TotalCharges'])
fig.add_hline(y=0.5, line_dash="dash", line_color="red", annotation_text="Threshold (0.5)")
fig.update_layout(height=500)
fig.show()

In [ ]:
# Monthly Charges vs Churn Probability
fig = px.box(df, x=df['Predicted_Churn'].map({0: 'No Churn', 1: 'Churn'}), 
             y='MonthlyCharges', color=df['Predicted_Churn'].map({0: 'No Churn', 1: 'Churn'}),
             color_discrete_map={'No Churn': '#2ecc71', 'Churn': '#e74c3c'},
             title='Monthly Charges Distribution by Prediction',
             labels={'x': 'Prediction', 'MonthlyCharges': 'Monthly Charges ($)'})
fig.update_layout(height=400)
fig.show()

## Step 7: Prediction Results Table

In [ ]:
# Summary table
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("=" * 60)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 60)
print(f"\nOverall Accuracy: {accuracy_score(df['Churn'], df['Predicted_Churn']):.4f}")
print(f"\n{classification_report(df['Churn'], df['Predicted_Churn'], target_names=['No Churn', 'Churn'])}")

# Confusion matrix heatmap
cm = confusion_matrix(df['Churn'], df['Predicted_Churn'])
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                labels=dict(x='Predicted', y='Actual', color='Count'),
                x=['No Churn', 'Churn'], y=['No Churn', 'Churn'],
                title='Confusion Matrix')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Top 20 high-risk customers table
high_risk = df.nlargest(20, 'Churn_Probability')[[
    'CustomerID', 'Gender', 'SeniorCitizen', 'Tenure', 'MonthlyCharges',
    'Contract', 'PaymentMethod', 'Churn_Probability', 'Predicted_Churn'
]].copy()
high_risk['Risk_Level'] = high_risk['Churn_Probability'].apply(
    lambda x: '🔴 Critical' if x > 0.8 else ('🟡 High' if x > 0.6 else '🟢 Medium')
)

print("\n" + "=" * 60)
print("TOP 20 HIGH-RISK CUSTOMERS")
print("=" * 60)
high_risk

In [ ]:
# Interactive risk table
fig = go.Figure(data=[go.Table(
    header=dict(
        values=['Customer ID', 'Gender', 'Tenure', 'Monthly Charges', 'Contract', 'Churn Prob', 'Risk'],
        fill_color='#2c3e50',
        font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[
            high_risk['CustomerID'], high_risk['Gender'], high_risk['Tenure'],
            high_risk['MonthlyCharges'].round(2), high_risk['Contract'],
            high_risk['Churn_Probability'].round(4), high_risk['Risk_Level']
        ],
        fill_color=[['#f8d7da' if p > 0.8 else '#fff3cd' if p > 0.6 else '#d4edda' 
                     for p in high_risk['Churn_Probability']]]*7,
        align='center'
    )
)])
fig.update_layout(title='High-Risk Customers Dashboard', height=600)
fig.show()

## Step 8: Create Streamlit Dashboard File

In [ ]:
streamlit_code = '''import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import requests
import pickle

st.set_page_config(page_title="Bank Churn Dashboard", layout="wide")
st.title("🏦 Bank Customer Churn Prediction Dashboard")

# Load data and model
@st.cache_data
def load_data():
    df = pd.read_csv("Bank_Churn_Classification_Dataset.csv", index_col=0)
    with open("model_artifacts/churn_model.pkl", "rb") as f:
        model = pickle.load(f)
    with open("model_artifacts/scaler.pkl", "rb") as f:
        scaler = pickle.load(f)
    with open("model_artifacts/label_encoders.pkl", "rb") as f:
        encoders = pickle.load(f)
    
    df_proc = df.drop("CustomerID", axis=1).copy()
    for col in encoders:
        if col in df_proc.columns:
            df_proc[col] = encoders[col].transform(df_proc[col])
    X = df_proc.drop("Churn", axis=1)
    X_scaled = scaler.transform(X)
    df["Churn_Probability"] = model.predict_proba(X_scaled)[:, 1]
    df["Predicted_Churn"] = model.predict(X_scaled)
    return df

df = load_data()

# Metrics
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Customers", f"{len(df):,}")
col2.metric("Predicted Churn", f"{df['Predicted_Churn'].sum():,}")
col3.metric("Churn Rate", f"{df['Predicted_Churn'].mean()*100:.1f}%")
col4.metric("Avg Churn Prob", f"{df['Churn_Probability'].mean():.3f}")

# Charts
col1, col2 = st.columns(2)
with col1:
    fig = px.pie(df, names=df["Predicted_Churn"].map({0: "No Churn", 1: "Churn"}), title="Prediction Distribution")
    st.plotly_chart(fig, use_container_width=True)
with col2:
    fig = px.histogram(df, x="Churn_Probability", nbins=50, title="Churn Probability Distribution")
    st.plotly_chart(fig, use_container_width=True)

# High risk table
st.subheader("🔴 High-Risk Customers")
high_risk = df.nlargest(20, "Churn_Probability")
st.dataframe(high_risk[["CustomerID", "Gender", "Tenure", "MonthlyCharges", "Contract", "Churn_Probability"]])
'''

with open('dashboard.py', 'w') as f:
    f.write(streamlit_code)

print("Streamlit dashboard created: dashboard.py")
print("Run with: streamlit run dashboard.py")
print("\n✅ Frontend visualization completed!")